In [1]:
from IPython.display import Image
Image("images\strukturExample.png");

# Workflow for the EnergyLand energy system

In this application of the FINE framework, a 1-node energy system is modeled and optimized.

The workflow is structures as follows:
1. Required packages are imported and the input data path is set
2. An energy system model instance is created
3. Commodity sources are added to the energy system model
4. Commodity conversion components are added to the energy system model
5. Commodity storages are added to the energy system model
6. Commodity sinks are added to the energy system model
7. The energy system model is optimized
8. Selected optimization results are presented

![Structure of EnergyLand](images\strukturExample.png)

# 1. Import packages

The FINE framework is imported which provides the required classes and functions for modeling the energy system.
The working directory and the underlying excelfile which provides some of the input data is imported.

In [2]:
import FINE as fn
from getData import getData
import pandas as pd
import os
cwd = os.getcwd()
data = getData()

# 2. Set up energy system model instance

The structure of the energy system model is given by the considered locations, in this case we consider only one location (EnergyLand), commodities, the number of time steps as well as the hours per time step.

The commodities are specified by a unit, which can be given as an energy or mass unit per hour.

In [3]:
locations = {'EnergyLand'}
commodityUnitDict = {'electricity': r'GW$_{el}$', 'hydrogen': r'GW$_{H_{2},LHV}$', 
                     'nGas':r'GW$_{CH4}$', 'coal':r'GW$_{coal}$', 'PHeat': r'GW$_{Pheat}$',
                     'LTHeat':r'GW$_{LTHeat}$', 'CO2':r'kt$_{CO_{2}}$/h', 'pTransport': 'Mio pkm/h',
                     'fTransport': 'Mio tkm/h', 'crudeOil': r'GW$_{Oil}$', 'wood': r'GW$_{wood}$', 
                     'biowaste': r'GW$_{biowaste}$', 'bioslurry': r'GW$_{bioslurry}$',
                     'diesel': r'GW$_{diesel}$', 'biogas': r'GW$_{CH4}$', 'nGasImp': r'GW$_{CH4}$'}
commodities = {'electricity', 'hydrogen', 'nGas', 'coal', 'PHeat', 'LTHeat', 'CO2', 'pTransport', 
               'fTransport', 'crudeOil', 'wood', 'biowaste', 'bioslurry', 'diesel', 'nGasImp', 'biogas'}
numberOfTimeSteps=8760
hoursPerTimeStep=1

In [4]:
esM = fn.EnergySystemModel(locations=locations, commodities=commodities, numberOfTimeSteps=8760,
                           commodityUnitsDict=commodityUnitDict,
                           hoursPerTimeStep=1, costUnit='1e6 Euro', lengthUnit='km', verboseLogLevel=0)

# 3. Sources

Source components transfer a commodity from outside the system boundary of EnergyLand into the system.

## 3.1 Electricity sources

### Wind turbines

#### Onshore Wind Turbines

In [5]:
esM.add(fn.Source(esM=esM, name='Wind_Onshore', commodity='electricity', hasCapacityVariable=True,
                  operationRateMax=data['Wind_onshore, operationRateMax'],
                  capacityMax=data['Wind_onshore, capacityMax'],
                  investPerCapacity=1250, opexPerCapacity=1250*0.02, interestRate=0.08,
                  economicLifetime=20))

#### Offshore Wind Turbines

In [6]:
esM.add(fn.Source(esM=esM, name='Wind_Offshore', commodity='electricity', hasCapacityVariable=True,
                  operationRateMax=data['Wind_offshore, operationRateMax'],
                  capacityMax=data['Wind_offshore, capacityMax'],
                  investPerCapacity=2530, opexPerCapacity=2530*0.045, interestRate=0.08,
                  economicLifetime=20))

### Photovoltaic

In [7]:
esM.add(fn.Source(esM=esM, name='PV', commodity='electricity', hasCapacityVariable=True,
                  operationRateMax=data['PV, operationRateMax'],
                  capacityMax=data['PV, capacityMax'],
                  investPerCapacity=800, opexPerCapacity=800*0.019, interestRate=0.08,
                  economicLifetime=20))

### Electricity import

In [8]:
esM.add(fn.Source(esM=esM, name='el_Import', commodity='electricity', hasCapacityVariable=False,
                  operationRateMax=data['el_Import, operationRateMax']))

## 3.2 Hydrogen source

In [9]:
esM.add(fn.Source(esM=esM, name='H2_Import', commodity='hydrogen', hasCapacityVariable=False,
                  operationRateMax=data['H2_Import, operationRateMax'],
                  commodityCost=0.132))

## 3.3 Coal source

In [10]:
esM.add(fn.Source(esM=esM, name='CoalSource', commodity='coal', hasCapacityVariable=False,
                  commodityCost=0.021))

## 3.4 Crude Oil source

In [11]:
esM.add(fn.Source(esM=esM, name='CrudeOilSource', commodity='crudeOil', hasCapacityVariable=False,
                  commodityCost=0.036))

## 3.5 Natural gas source

In [12]:
esM.add(fn.Source(esM=esM, name='nGasSource', commodity='nGasImp', hasCapacityVariable=False,
                  commodityCost=0.0256))

## 3.6 Biomass sources

#### Wood Source

In [13]:
esM.add(fn.Source(esM=esM, name='WoodSource', commodity='wood', hasCapacityVariable=True,
                  capacityMax=data['wood_source, capacityMax'],
                  commodityCost=0.028))

#### Biowaste Source

In [14]:
esM.add(fn.Source(esM=esM, name='BiowasteSource', commodity='biowaste', hasCapacityVariable=True,
                  capacityMax=data['biowaste_source, capacityMax'],
                  commodityCost=0.07))

#### Bioslurry Source

In [15]:
esM.add(fn.Source(esM=esM, name='BioslurrySource', commodity='bioslurry', hasCapacityVariable=True,
                  capacityMax=data['bioslurry_source, capacityMax'],
                  commodityCost=0.07))

# 4. Conversion components

These are the components which can transfer one commodity into another one.

## 4.1 Biomas to biogas

### Bioslurry to Biogas

In [16]:
esM.add(fn.Conversion(esM=esM, name='bioslurry-biogas', physicalUnit= r'GW$_{CH4}$',
                      commodityConversionFactors={'bioslurry':-1, 'biogas':1},
                      hasCapacityVariable=False))

### Biowaste to Biogas

In [17]:
esM.add(fn.Conversion(esM=esM, name='biowaste-biogas', physicalUnit= r'GW$_{CH4}$',
                      commodityConversionFactors={'biowaste':-1, 'biogas':1},
                      hasCapacityVariable=False))

## 4.2 Methane Slip (Virtual conversion)

In [18]:
methaneSlip=0.1
esM.add(fn.Conversion(esM=esM, name='CH4Slip', physicalUnit= r'GW$_{CH4}$',
                      commodityConversionFactors={'nGasImp':-1, 'nGas':1, 'CO2':methaneSlip*2.014},
                      hasCapacityVariable=False))

## 4.3 Biogas to Methane (Virtual conversion)

In [19]:
esM.add(fn.Conversion(esM=esM, name='biogas-nGas', physicalUnit= r'GW$_{CH4}$',
                      commodityConversionFactors={'biogas':-1, 'nGas':1, 'CO2':-0.2014},
                      hasCapacityVariable=True, opexPerOperation=0.0003,
                      investPerCapacity=343, opexPerCapacity=343*0.025, interestRate=0.08,
                      economicLifetime=15))

## 4.4 Transport

### Batterie Electric Vehicle

#### BEV Car

In [20]:
esM.add(fn.Conversion(esM=esM, name='BEV_PCar', physicalUnit= r'Mio pkm/h',
                      commodityConversionFactors={'electricity':-1/7.676226, 'pTransport':1},
                      hasCapacityVariable=True, 
                      investPerCapacity=15694, opexPerCapacity=15694*0.009, interestRate=0.08,
                      economicLifetime=12))

#### BEV Truck

In [21]:
esM.add(fn.Conversion(esM=esM, name='BEV_Truck', physicalUnit= r'Mio tkm/h',
                      commodityConversionFactors={'electricity':-1/11.401, 'fTransport':1},
                      hasCapacityVariable=True, 
                      investPerCapacity=4304, opexPerCapacity=4304*0.009, interestRate=0.08,
                      economicLifetime=15))

### Fuel Cell Electric Vehicle

#### FCEV Car

In [22]:
esM.add(fn.Conversion(esM=esM, name='FCEV_PCar', physicalUnit= r'Mio pkm/h',
                      commodityConversionFactors={'hydrogen':-1/4.7472, 'pTransport':1},
                      hasCapacityVariable=True, 
                      investPerCapacity=15694, opexPerCapacity=15694*0.009, interestRate=0.08,
                      economicLifetime=12))

#### FCEV Truck

In [23]:
esM.add(fn.Conversion(esM=esM, name='FCEV_Truck', physicalUnit= r'Mio tkm/h',
                      commodityConversionFactors={'hydrogen':-1/8.251, 'fTransport':1},
                      hasCapacityVariable=True,
                      investPerCapacity=4283, opexPerCapacity=4283*0.009, interestRate=0.08,
                      economicLifetime=15))

### Fossil Vehicles

#### Fossil Car

In [24]:
esM.add(fn.Conversion(esM=esM, name='FossilCar', physicalUnit= r'Mio pkm/h',
                      commodityConversionFactors={'diesel':-1/3.1308, 'pTransport':1},
                      hasCapacityVariable=True, 
                      investPerCapacity=15694, opexPerCapacity=15694*0.016, interestRate=0.08,
                      economicLifetime=12))

#### Fossil Truck

In [25]:
esM.add(fn.Conversion(esM=esM, name='FossilTruck', physicalUnit= r'Mio tkm/h',
                      commodityConversionFactors={'diesel':-1/7.938, 'fTransport':1},
                      hasCapacityVariable=True, 
                      investPerCapacity=3342, opexPerCapacity=3342*0.016, interestRate=0.08,
                      economicLifetime=15))

## 4.5 Diesel Refinery

In [26]:
esM.add(fn.Conversion(esM=esM, name='DieselRef', physicalUnit= r'GW$_{diesel}$',
                      commodityConversionFactors={'crudeOil':-1/0.364, 'diesel':1, 'CO2':0.725},
                      hasCapacityVariable=True, 
                      investPerCapacity=1/0.364, opexPerCapacity=(1/0.364)*0.001, interestRate=0.08,
                      economicLifetime=20))

## 4.6 Power Plants

### Combined Cycle Gas Turbine

#### Natural Gas CCGT

In [27]:
esM.add(fn.Conversion(esM=esM, name='CCGT plants (NGas)', physicalUnit=r'GW$_{el}$',
                      commodityConversionFactors={'electricity':1, 'nGas':-1/0.65, 'CO2':0.31},
                      hasCapacityVariable=True,investPerCapacity=850, 
                      opexPerCapacity=850*0.03, opexPerOperation=0.002, interestRate=0.08,
                      economicLifetime=30))

#### H2 CCGT

In [28]:
esM.add(fn.Conversion(esM=esM, name='CCGT plants (hydrogen)', physicalUnit=r'GW$_{el}$',
                      commodityConversionFactors={'electricity':1, 'hydrogen':-1/0.6},
                      hasCapacityVariable=True, investPerCapacity=760, 
                      opexPerCapacity=760*0.014, opexPerOperation=0.002, interestRate=0.08,
                      economicLifetime=20))

### Fuel cell

In [29]:
esM.add(fn.Conversion(esM=esM, name='LS-SOFC', physicalUnit=r'GW$_{el}$',
                      commodityConversionFactors={'electricity':1, 'hydrogen':-1/0.7, 'LTHeat':0.25/0.7},
                      hasCapacityVariable=True, investPerCapacity=1210, 
                      opexPerCapacity=1210*0.008, interestRate=0.08,
                      economicLifetime=20))

### Coal power plant

In [30]:
esM.add(fn.Conversion(esM=esM, name='CoalPP', physicalUnit=r'GW$_{el}$',
                      commodityConversionFactors={'electricity':1, 'coal':-1/0.5, 'CO2':0.674},
                      hasCapacityVariable=True, opexPerOperation=0.0015,
                      investPerCapacity=1450, opexPerCapacity=1450*0.026, interestRate=0.08,
                      economicLifetime=40))

### Combined Heat and Power Plants

#### Coal CHP

In [31]:
esM.add(fn.Conversion(esM=esM, name='CoalCHP', physicalUnit=r'GW$_{el}$',
                      commodityConversionFactors={'electricity':1, 'LTHeat':0.51/0.38, 'coal':-1/0.38, 'CO2':0.886},
                      hasCapacityVariable=True, opexPerOperation=0.0051,
                      investPerCapacity=1847, opexPerCapacity=1847*0.027, interestRate=0.08,
                      economicLifetime=35))

#### Wood CHP

In [32]:
esM.add(fn.Conversion(esM=esM, name='WoodCHP', physicalUnit=r'GW$_{el}$',
                      commodityConversionFactors={'electricity':1, 'LTHeat':0.826/0.291, 'wood':-1/0.291},
                      hasCapacityVariable=True, opexPerOperation=0.0038,
                      investPerCapacity=3000, opexPerCapacity=3000*0.029, interestRate=0.08,
                      economicLifetime=25))

#### Natural Gas CHP

In [33]:
esM.add(fn.Conversion(esM=esM, name='nGasCHP', physicalUnit=r'GW$_{el}$',
                      commodityConversionFactors={'electricity':1, 'LTHeat':0.5/0.35, 'nGas':-1/0.35, 'CO2':0.575},
                      hasCapacityVariable=True, opexPerOperation=0.0015,
                      investPerCapacity=666, opexPerCapacity=666*0.041, interestRate=0.08,
                      economicLifetime=30))

#### Biogas CHP

In [34]:
esM.add(fn.Conversion(esM=esM, name='BioGasCHP', physicalUnit=r'GW$_{el}$',
                      commodityConversionFactors={'electricity':1, 'LTHeat':1, 'biogas':-1/0.47},
                      hasCapacityVariable=True, opexPerOperation=0.008,
                      investPerCapacity=850, opexPerCapacity=850*0.01, interestRate=0.08,
                      economicLifetime=25))

#### H2 CHP

In [35]:
esM.add(fn.Conversion(esM=esM, name='H2CHP', physicalUnit=r'GW$_{el}$',
                      commodityConversionFactors={'electricity':1, 'LTHeat':0.41/0.49, 'hydrogen':-1/0.49},
                      hasCapacityVariable=True, opexPerOperation=0.0006,
                      investPerCapacity=715, opexPerCapacity=715*0.001, interestRate=0.08,
                      economicLifetime=20))

## 4.7 Thermal power plants

#### Oil Boiler

In [36]:
esM.add(fn.Conversion(esM=esM, name='oilBoiler', physicalUnit=r'GW$_{LTHeat}$',
                      commodityConversionFactors={'crudeOil':-1/0.96, 'LTHeat':1, 'CO2':0.275},
                      hasCapacityVariable=True, 
                      investPerCapacity=330, opexPerCapacity=330*0.041, interestRate=0.08,
                      economicLifetime=20))

#### Gas Boiler

In [37]:
esM.add(fn.Conversion(esM=esM, name='gasBoiler', physicalUnit=r'GW$_{LTHeat}$',
                      commodityConversionFactors={'nGas':-1/0.96, 'LTHeat':1, 'CO2':0.21},
                      hasCapacityVariable=True, 
                      investPerCapacity=330, opexPerCapacity=330*0.012, interestRate=0.08,
                      economicLifetime=20))

#### H2 Boiler

In [38]:
esM.add(fn.Conversion(esM=esM, name='H2Boiler', physicalUnit=r'GW$_{LTHeat}$',
                      commodityConversionFactors={'hydrogen':-1/0.98, 'LTHeat':1},
                      hasCapacityVariable=True, 
                      investPerCapacity=655, opexPerCapacity=655*0.01, interestRate=0.08,
                      economicLifetime=20))

### Heat pump

In [39]:
esM.add(fn.Conversion(esM=esM, name='Heatpump', physicalUnit=r'GW$_{LTHeat}$',
                      commodityConversionFactors={'electricity':-1/0.45, 'LTHeat':1},
                      hasCapacityVariable=True, 
                      investPerCapacity=725, opexPerCapacity=725*0.02, interestRate=0.08,
                      economicLifetime=20))

### Heating rod

In [40]:
esM.add(fn.Conversion(esM=esM, name='Heating rod', physicalUnit=r'GW$_{LTHeat}$',
                      commodityConversionFactors={'electricity':-1/0.99, 'LTHeat':1},
                      hasCapacityVariable=True, 
                      investPerCapacity=800, opexPerCapacity=800*0.0125, interestRate=0.08,
                      economicLifetime=30))

### Electrode boiler

In [41]:
esM.add(fn.Conversion(esM=esM, name='electrode boiler', physicalUnit=r'GW$_{Pheat}$',
                      commodityConversionFactors={'electricity':-1/0.99, 'PHeat':1},
                      hasCapacityVariable=True, 
                      investPerCapacity=140, opexPerCapacity=140*0.02, interestRate=0.08,
                      economicLifetime=20))

### Stove

In [42]:
esM.add(fn.Conversion(esM=esM, name='woood Stove', physicalUnit=r'GW$_{LTHeat}$',
                      commodityConversionFactors={'wood':-1/0.75, 'LTHeat':1},
                      hasCapacityVariable=True, 
                      investPerCapacity=775, opexPerCapacity=775*0.06, interestRate=0.08,
                      economicLifetime=20))

## 4.8 Electrolyzer

In [43]:
esM.add(fn.Conversion(esM=esM, name='electroylzer', physicalUnit=r'GW$_{H_{2},LHV}$',
                      commodityConversionFactors={'electricity':-1/0.7, 'hydrogen':1},
                      hasCapacityVariable=True, 
                      investPerCapacity=500, opexPerCapacity=500*0.03, interestRate=0.08,
                      economicLifetime=10))

# 5. Storages

The storages which can be used by the EnergyLand model are constructed.

## Lithium ion batteries

In [44]:
esM.add(fn.Storage(esM=esM, name='Li-ion batteries', commodity='electricity',
                   hasCapacityVariable=True, chargeEfficiency=0.99,
                   dischargeEfficiency=0.99, selfDischarge=0.004,
                   doPreciseTsaModeling=False,investPerCapacity=120, 
                   opexPerCapacity=120*0.014, opexPerChargeOperation=0.0001, 
                   interestRate=0.08, economicLifetime=10))

## Hydrogen filled salt caverns

In [45]:
esM.add(fn.Storage(esM=esM, name='H2Storage', commodity='hydrogen',
                   hasCapacityVariable=True, chargeEfficiency=0.98,
                   dischargeEfficiency=0.998, doPreciseTsaModeling=False,
                   investPerCapacity=362, opexPerCapacity=362*0.02, opexPerChargeOperation=0.0001, 
                   interestRate=0.08, economicLifetime=40))

## Heat storage

In [46]:
esM.add(fn.Storage(esM=esM, name='LTHeatstorage', commodity='LTHeat',
                   hasCapacityVariable=True, chargeEfficiency=0.95,
                   dischargeEfficiency=0.95, selfDischarge=0.0003,
                   chargeRate=1, dischargeRate=1, doPreciseTsaModeling=False,
                   investPerCapacity=147, opexPerCapacity=147*0.01, opexPerChargeOperation=0.0001,
                   interestRate=0.08, economicLifetime=20))

# 6. Sinks

Electricity, heat and transport demand are set in the following components.

## Electricity demand

In [47]:
eDemand=516
esM.add(fn.Sink(esM=esM, name='Electricity demand', commodity='electricity',
                hasCapacityVariable=False, operationRateFix=data['Electricity demand, operationRateFix']*eDemand))

## Passenger Transportation demand

In [48]:
pTdemand=867
esM.add(fn.Sink(esM=esM, name='pT_demand', commodity='pTransport',
                hasCapacityVariable=False, operationRateFix=data['T_demand, operationRateFix']*pTdemand))

## Freight Transportation demand

In [49]:
fTdemand=945.5
esM.add(fn.Sink(esM=esM, name='fT_demand', commodity='fTransport',
                hasCapacityVariable=False, operationRateFix=data['T_demand, operationRateFix']*fTdemand))

## Heat demand

### Process heat demand

In [50]:
pHeatDemand=423.75
esM.add(fn.Sink(esM=esM, name='PHeat_demand', commodity='PHeat',
                hasCapacityVariable=False, operationRateFix=data['pHeat_demand, operationRateFix']*pHeatDemand))

### Low temperature residential heat demand

In [51]:
LTHeatDemand=560.8
esM.add(fn.Sink(esM=esM, name='LTHeat_demand', commodity='LTHeat',
                hasCapacityVariable=False, operationRateFix=data['LtHeat_demand, operationRateFix']*LTHeatDemand))

## Environment

The CO2 limit is set in this component.

In [52]:
CO2limit=210000
esM.add(fn.Sink(esM=esM, name='CO2 to environment', commodity='CO2', commodityLimitID='CO2_cap',
                hasCapacityVariable=False, yearlyLimit=CO2limit))

# 7. Optimization of EnergyLand

In [54]:
myopicResults = fn.optimizeSimpleMyopic(esM, startYear=2020, endYear=2050, nbOfRepresentedYears=10,
                    timeSeriesAggregation=True, numberOfTypicalPeriods = 12, numberOfTimeStepsPerPeriod=24,
                    logFileName='', threads=3, solver='gurobi', timeLimit=None, 
                    optimizationSpecs='', warmstart=False, 
                    CO2Reference=366000, CO2ReductionTargets=[50,70,90,100], saveResults=False, trackESMs=True)

Number of optimization runs:  4
Number of years represented by one optimization:  10

Clustering time series data with 12 typical periods and 24 time steps per period...
		(0.2845 sec)

Time series aggregation specifications:
Number of typical periods:12, number of time steps per periods:24

Declaring sets, variables and constraints for SourceSinkModel
	declaring sets... 
	declaring variables... 
	declaring constraints... 
		(0.1694 sec)

Declaring sets, variables and constraints for ConversionModel
	declaring sets... 
	declaring variables... 
	declaring constraints... 
		(0.2359 sec)

Declaring sets, variables and constraints for StorageModel
	declaring sets... 
	declaring variables... 
	declaring constraints... 
		(0.2232 sec)

Declaring shared potential constraint...
		(0.0000 sec)

Declaring commodity balances...
		(0.2473 sec)

Declaring objective function...
		(0.2032 sec)

Using license file C:\Users\t.schoeb\gurobi.lic
Academic license - for non-commercial use only
Read LP form

Ordering time: 0.23s

Barrier statistics:
 Dense cols : 70
 Free vars  : 180
 AA' NZ     : 2.381e+05
 Factor NZ  : 1.238e+06 (roughly 23 MBytes of memory)
 Factor Ops : 2.446e+08 (less than 1 second per iteration)
 Threads    : 2

                  Objective                Residual
Iter       Primal          Dual         Primal    Dual     Compl     Time
   0   3.12056655e+07 -1.33561641e+07  2.12e+05 6.73e+01  2.92e+05     0s
   1   2.94974290e+07 -2.21183624e+07  1.48e+05 6.93e+01  1.28e+05     0s
   2   2.09275009e+07 -2.73555719e+07  7.46e+04 1.16e+01  5.42e+04     0s
   3   9.14110162e+06 -2.56447699e+07  9.24e+03 1.20e+00  7.36e+03     1s
   4   4.70280105e+06 -1.78723908e+07  2.19e+03 3.35e-01  2.05e+03     1s
   5   3.53998844e+06 -9.69461610e+06  1.30e+03 9.75e-02  1.01e+03     1s
   6   1.86853582e+06 -3.47775787e+06  3.45e+02 1.30e-02  2.74e+02     1s
   7   1.27172027e+06 -1.39764734e+06  1.52e+02 1.92e-03  1.15e+02     1s
   8   1.08246479e+06 -6.96520241e+05  1.02e+02 1.8

  14   6.31913338e+05  5.81265935e+05  3.05e+00 6.32e-04  2.03e+00     1s
  15   6.21526078e+05  5.95037944e+05  1.65e+00 4.48e-04  1.04e+00     1s
  16   6.13636366e+05  6.01155758e+05  7.42e-01 2.08e-04  4.82e-01     1s
  17   6.11289689e+05  6.04075575e+05  4.94e-01 7.89e-05  2.79e-01     1s
  18   6.09094110e+05  6.05348532e+05  2.72e-01 9.18e-05  1.44e-01     1s
  19   6.07978705e+05  6.05760055e+05  1.64e-01 9.75e-05  8.47e-02     1s
  20   6.07778874e+05  6.05956499e+05  1.42e-01 1.01e-04  6.94e-02     1s
  21   6.07609898e+05  6.05995925e+05  1.26e-01 1.11e-04  6.14e-02     1s
  22   6.07030136e+05  6.06099389e+05  6.85e-02 1.31e-04  3.51e-02     1s
  23   6.06684855e+05  6.06175628e+05  3.52e-02 1.21e-04  1.90e-02     1s
  24   6.06511448e+05  6.06224948e+05  1.92e-02 8.41e-05  1.06e-02     1s
  25   6.06431997e+05  6.06253149e+05  1.17e-02 5.34e-05  6.59e-03     1s
  26   6.06384990e+05  6.06265484e+05  7.67e-03 3.85e-05  4.40e-03     1s
  27   6.06369191e+05  6.06271753e+05 

  33   8.87571224e+05  8.87482939e+05  1.89e-04 4.11e-05  3.71e-03     2s
  34   8.87556186e+05  8.87485911e+05  1.40e-04 4.13e-05  2.93e-03     2s
  35   8.87513625e+05  8.87493195e+05  2.61e-05 4.60e-05  8.03e-04     2s
  36   8.87504744e+05  8.87494550e+05  1.17e-05 3.27e-05  3.89e-04     2s
  37   8.87499691e+05  8.87495891e+05  3.81e-06 1.29e-05  1.40e-04     2s
  38   8.87498263e+05  8.87496498e+05  1.64e-06 7.63e-06  6.39e-05     2s
  39   8.87497894e+05  8.87496781e+05  1.13e-06 3.39e-06  4.11e-05     2s
  40   8.87497668e+05  8.87496846e+05  8.19e-07 2.54e-06  3.03e-05     2s
  41   8.87497461e+05  8.87496907e+05  5.39e-07 1.76e-06  2.03e-05     2s
  42   8.87497277e+05  8.87496982e+05  2.84e-07 7.46e-07  1.08e-05     2s
  43   8.87497197e+05  8.87497014e+05  1.81e-07 3.26e-07  6.74e-06     2s
  44   8.87497172e+05  8.87497021e+05  1.53e-07 2.63e-07  5.59e-06     2s
  45   8.87497142e+05  8.87497028e+05  1.13e-07 1.96e-07  4.19e-06     2s
  46   8.87497113e+05  8.87497034e+05 

# 8. Results

In [55]:
esM2020 = myopicResults['ESM_2020']
esM2030 = myopicResults['ESM_2030']
esM2040 = myopicResults['ESM_2040']
esM2050 = myopicResults['ESM_2050']

In [56]:
esM2020.getOptimizationSummary("SourceSinkModel", outputLevel=2).loc['PV']

,,EnergyLand
Property,Unit,
TAC,[1e6 Euro/a],23590.4
capacity,[GW$_{el}$],244
capexCap,[1e6 Euro/a],19881.6
invest,[1e6 Euro],195200
operation,[GW$_{el}$*h/a],276865
opexCap,[1e6 Euro/a],3708.8


In [57]:
i = ['PV_stock_2020' , 'PV']
esM2030.getOptimizationSummary("SourceSinkModel", outputLevel=2).loc[i]

EnergyLand
Component     Property  Unit                      
PV            TAC       [1e6 Euro/a]       17534.9
              capacity  [GW$_{el}$]        181.368
              capexCap  [1e6 Euro/a]       14778.2
              invest    [1e6 Euro]          145094
              operation [GW$_{el}$*h/a]     159993
              opexCap   [1e6 Euro/a]       2756.79
PV_stock_2020 TAC       [1e6 Euro/a]       23590.4
              capacity  [GW$_{el}$]            244
              capexCap  [1e6 Euro/a]       19881.6
              invest    [1e6 Euro]          195200
              operation [GW$_{el}$*h/a]     287410
              opexCap   [1e6 Euro/a]        3708.8

In [58]:
i.extend(['PV_stock_2030'])
print(i)
esM2040.getOptimizationSummary("SourceSinkModel", outputLevel=2).loc[i]

['PV_stock_2020', 'PV', 'PV_stock_2030']


EnergyLand
Component     Property  Unit                      
PV            TAC       [1e6 Euro/a]       23590.4
              capacity  [GW$_{el}$]            244
              capexCap  [1e6 Euro/a]       19881.6
              invest    [1e6 Euro]          195200
              operation [GW$_{el}$*h/a]     262766
              opexCap   [1e6 Euro/a]        3708.8
PV_stock_2030 TAC       [1e6 Euro/a]       17534.9
              capacity  [GW$_{el}$]        181.368
              capexCap  [1e6 Euro/a]       14778.2
              invest    [1e6 Euro]          145094
              operation [GW$_{el}$*h/a]     225327
              opexCap   [1e6 Euro/a]       2756.79

In [59]:
i.extend(['PV_stock_2040'])
esM2050.getOptimizationSummary("SourceSinkModel", outputLevel=2).loc[i]

EnergyLand
Component     Property  Unit                      
PV            TAC       [1e6 Euro/a]       23590.4
              capacity  [GW$_{el}$]            244
              capexCap  [1e6 Euro/a]       19881.6
              invest    [1e6 Euro]          195200
              operation [GW$_{el}$*h/a]    55241.2
              opexCap   [1e6 Euro/a]        3708.8
PV_stock_2040 TAC       [1e6 Euro/a]       23590.4
              capacity  [GW$_{el}$]            244
              capexCap  [1e6 Euro/a]       19881.6
              invest    [1e6 Euro]          195200
              operation [GW$_{el}$*h/a]     242403
              opexCap   [1e6 Euro/a]        3708.8

In [60]:
esM2020.getOptimizationSummary("ConversionModel", outputLevel=2).loc['CCGT plants (NGas)']

,,EnergyLand
Property,Unit,
TAC,[1e6 Euro/a],5712.99
capacity,[GW$_{el}$],54.8474
capexCap,[1e6 Euro/a],4141.16
invest,[1e6 Euro],46620.3
operation,[GW$_{el}$*h/a],86610.4
opexCap,[1e6 Euro/a],1398.61
opexOp,[1e6 Euro/a],173.221


In [61]:
i = ['CCGT plants (NGas)','CCGT plants (NGas)_stock_2020']
esM2030.getOptimizationSummary("ConversionModel", outputLevel=2).loc[i]

EnergyLand
Component                     Property  Unit                      
CCGT plants (NGas)_stock_2020 TAC       [1e6 Euro/a]       5649.82
                              capacity  [GW$_{el}$]        54.8474
                              capexCap  [1e6 Euro/a]       4141.16
                              invest    [1e6 Euro]         46620.3
                              operation [GW$_{el}$*h/a]    55026.6
                              opexCap   [1e6 Euro/a]       1398.61
                              opexOp    [1e6 Euro/a]       110.053

In [62]:
i.extend(['CCGT plants (NGas)_stock_2030'])
esM2040.getOptimizationSummary("ConversionModel", outputLevel=2).loc[i]

EnergyLand
Component                     Property  Unit                      
CCGT plants (NGas)_stock_2020 TAC       [1e6 Euro/a]       5582.63
                              capacity  [GW$_{el}$]        54.8474
                              capexCap  [1e6 Euro/a]       4141.16
                              invest    [1e6 Euro]         46620.3
                              operation [GW$_{el}$*h/a]      21431
                              opexCap   [1e6 Euro/a]       1398.61
                              opexOp    [1e6 Euro/a]       42.8621

In [63]:
i.extend(['CCGT plants (NGas)_stock_2040'])
esM2050.getOptimizationSummary("ConversionModel", outputLevel=2).loc[i]

,,,EnergyLand
Component,Property,Unit,


In [70]:
i.extend(['Wind_Onshore_stock_2030'])
esM2050.getOptimizationSummary("SourceSinkModel", outputLevel=2).loc['Wind_Onshore']

,,EnergyLand
Property,Unit,
TAC,[1e6 Euro/a],94435.5
capacity,[GW$_{el}$],620
capexCap,[1e6 Euro/a],78935.5
invest,[1e6 Euro],775000
operation,[GW$_{el}$*h/a],230220
opexCap,[1e6 Euro/a],15500
